In [1]:
# ==========================================
# PYTORCH HEALTH DIAGNOSIS SYSTEM
# Breast Cancer Prediction using Deep Learning
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

print("=" * 50)
print("PYTORCH HEALTH DIAGNOSIS SYSTEM")
print("=" * 50)

# ==========================================
# 1. DATA MODELLING
# ==========================================

print("\n[1/4] Loading Medical Dataset...")

data = load_breast_cancer()

X = data.data
y = data.target

print(f"Number of Samples : {len(X)}")
print(f"Number of Features: {X.shape[1]}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

y_train = torch.FloatTensor(y_train).view(-1, 1)
y_test = torch.FloatTensor(y_test).view(-1, 1)

# ==========================================
# 2. MACHINE LEARNING ALGORITHM
# ==========================================

print("\n[2/4] Building Neural Network...")

class DiagnosisModel(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

model = DiagnosisModel(X_train.shape[1])

criterion = nn.BCELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ==========================================
# 3. TRAINING
# ==========================================

print("\n[3/4] Training Model...")

epochs = 100

for epoch in range(epochs):

    outputs = model(X_train)

    loss = criterion(
        outputs,
        y_train
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {loss.item():.4f}"
        )

# ==========================================
# 4. PREDICTIVE SYSTEM
# ==========================================

print("\n[4/4] Evaluating Predictive System...")

model.eval()

with torch.no_grad():

    predictions = model(X_test)

    predicted_labels = (
        predictions > 0.5
    ).float()

accuracy = accuracy_score(
    y_test,
    predicted_labels
)

print("\n" + "=" * 50)
print("SYSTEM RESULTS")
print("=" * 50)

print(
    f"Predictive Accuracy: "
    f"{accuracy * 100:.2f}%"
)

sample_prediction = predicted_labels[0].item()

diagnosis = (
    "Benign (Low Risk)"
    if sample_prediction == 1
    else "Malignant (High Risk)"
)

print(
    f"Patient #1 Diagnosis: {diagnosis}"
)

cm = confusion_matrix(
    y_test,
    predicted_labels
)

print("\nConfusion Matrix:")
print(cm)

PYTORCH HEALTH DIAGNOSIS SYSTEM

[1/4] Loading Medical Dataset...
Number of Samples : 569
Number of Features: 30

[2/4] Building Neural Network...

[3/4] Training Model...
Epoch [10/100] Loss: 0.6880
Epoch [20/100] Loss: 0.6306
Epoch [30/100] Loss: 0.5554
Epoch [40/100] Loss: 0.4666
Epoch [50/100] Loss: 0.3744
Epoch [60/100] Loss: 0.2932
Epoch [70/100] Loss: 0.2309
Epoch [80/100] Loss: 0.1862
Epoch [90/100] Loss: 0.1546
Epoch [100/100] Loss: 0.1318

[4/4] Evaluating Predictive System...

SYSTEM RESULTS
Predictive Accuracy: 96.49%
Patient #1 Diagnosis: Benign (Low Risk)

Confusion Matrix:
[[42  1]
 [ 3 68]]
